# Week 6 Assignment - PySpark (Databricks)

Objective: Understand Spark architecture and perform data processing using transformations, filtering, schema handling, optimized file formats, and Spark performance concepts, using a self-created orders dataset.


## Step 1: Import Libraries

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder.appName("Week6_Orders_Assignment").getOrCreate()


### Spark Architecture
Before getting into the code, it helps to know what actually happens when spark.read or any job runs. The Driver is the program that creates the SparkSession, builds the execution plan and hands out tasks. The Cluster Manager takes care of allocating resources (worker nodes) for the job, and the Executors are the ones running on those worker nodes that actually process the data and send results back to the driver.


### Client Mode vs Cluster Mode
Spark applications can run in Client mode or Cluster mode. In Client mode the driver runs on the machine that submitted the job, so it needs to stay connected until the job finishes. In Cluster mode the driver runs inside the cluster itself, which is more reliable for long running production jobs. A Databricks notebook session behaves like a managed version of this.


### Lazy Evaluation and DAG
Spark doesn't run a transformation the moment I write it. Things like filter() or select() just get added to a lineage graph (DAG) of steps. Only when I call an action such as show() or count() does Spark actually look at the whole DAG, optimize it, and execute it. That's why nothing really "runs" until an action shows up.


### Fault Tolerance
Because Spark keeps this lineage of transformations instead of copying data around, it doesn't need multiple backups to stay safe. If a worker node fails in the middle of a job, Spark can just recompute the lost partitions from the lineage graph instead of restarting the whole job.


### Shuffle
Operations like groupBy() or join() need matching keys to sit on the same executor, so Spark has to move data across the cluster to make that happen - this is called a shuffle. Shuffles are costly since they involve network and disk I/O, so I tried to filter the data down first before doing any groupBy later in this notebook.


## Step 2: About the Dataset
I created my own dataset (orders_dataset_week6.csv) instead of downloading one, so I have full control over nulls, duplicates, categories etc.
It has around 2000 rows and 13 columns (order_id, user_id, product_id, category, product_name, base_price, price, amount, quantity, region, priority, status, order_date).
The price column is kept as String on purpose, so I can practice casting it to Double later.




## Step 3: Reading the CSV file
Here I'm reading the CSV with header=True so the first row becomes column names, and inferSchema=True so Spark scans the data and guesses the datatypes on its own.


In [0]:
df = spark.read.csv("/Workspace/PySpark/orders_dataset_week6.csv", header=True, inferSchema=True)
df.show(5)


+--------+-------+----------+-----------+-------------+----------+-------+--------+--------+------+--------+----------+----------+
|order_id|user_id|product_id|   category| product_name|base_price|  price|  amount|quantity|region|priority|    status|order_date|
+--------+-------+----------+-----------+-------------+----------+-------+--------+--------+------+--------+----------+----------+
|       1| 1072.0|     P9935|Electronics|       Laptop|    3733.6| 3733.6| 14934.4|       4| North|     Low|Processing|2025-04-25|
|       2|   NULL|     P4257|Electronics|       Laptop|    559.11| 559.11| 2236.44|       4|  West|    High|Processing|2025-09-16|
|       3| 1358.0|     P5552|     Sports|Tennis Racket|   4066.21|4066.21| 4066.21|       1| South|    High| Cancelled|2025-03-23|
|       4| 1310.0|     P1711|Electronics|       Laptop|   1961.64|1961.64|11769.84|       6|  West|     Low| Completed|2025-06-26|
|       5| 1296.0|     P2139|  Groceries|         Rice|    2805.0| 2805.0| 28050.0|

## Step 4: Exploring the data a bit

In [0]:
df.printSchema()
print("Total columns:", len(df.columns))
print(df.columns)
print("Total rows:", df.count())
df.describe().show()


root
 |-- order_id: integer (nullable = true)
 |-- user_id: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)

Total columns: 13
['order_id', 'user_id', 'product_id', 'category', 'product_name', 'base_price', 'price', 'amount', 'quantity', 'region', 'priority', 'status', 'order_date']
Total rows: 2020
+-------+-----------------+------------------+----------+--------+------------+------------------+------------------+------------------+------------------+------+--------+----------+
|summary|         order_id|           user_id|product_id|category|product_name|        base_price| 

## Step 5: Reading the CSV with a custom schema
Instead of letting Spark guess the schema every time, I defined it manually below. inferSchema is convenient, but it actually makes Spark read the file twice (once to guess the types, once to load it), so an explicit schema is better once I already know my columns.


In [0]:
orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("base_price", DoubleType(), True),
    StructField("price", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("region", StringType(), True),
    StructField("priority", StringType(), True),
    StructField("status", StringType(), True),
    StructField("order_date", StringType(), True)
])

df_schema = spark.read.csv("/Workspace/PySpark/orders_dataset_week6.csv", header=True, schema=orders_schema)
df_schema.printSchema()


root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- price: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: string (nullable = true)



## Step 6: Renaming a column and casting a datatype
Next, I renamed product_name to item_name just to practice withColumnRenamed(), and cast the price column (which was loaded as a string) to Double so it can actually be used in calculations later.


In [0]:
df = df.withColumnRenamed("product_name", "item_name")
df = df.withColumn("price", col("price").cast(DoubleType()))

df.printSchema()
df.select("item_name", "price").show(5)


root
 |-- order_id: integer (nullable = true)
 |-- user_id: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)

+-------------+-------+
|    item_name|  price|
+-------------+-------+
|       Laptop| 3733.6|
|       Laptop| 559.11|
|Tennis Racket|4066.21|
|       Laptop|1961.64|
|         Rice| 2805.0|
+-------------+-------+
only showing top 5 rows


## Step 7: Adding a new column - final_price
Here I'm adding an 18% tax on top of base_price to get a final_price column.


In [0]:
df = df.withColumn("final_price", col("base_price") * 1.18)
df.select("base_price", "final_price").show(5)


+----------+------------------+
|base_price|       final_price|
+----------+------------------+
|    3733.6| 4405.647999999999|
|    559.11| 659.7497999999999|
|   4066.21|         4798.1278|
|   1961.64|         2314.7352|
|    2805.0|3309.8999999999996|
+----------+------------------+
only showing top 5 rows


## Step 8: Filtering - only Electronics orders
Selecting rows where category equals Electronics.


In [0]:
electronics_df = df.filter(col("category") == "Electronics")
electronics_df.show(5)
print("Electronics orders:", electronics_df.count())


+--------+-------+----------+-----------+---------+----------+-------+--------+--------+------+--------+----------+----------+------------------+
|order_id|user_id|product_id|   category|item_name|base_price|  price|  amount|quantity|region|priority|    status|order_date|       final_price|
+--------+-------+----------+-----------+---------+----------+-------+--------+--------+------+--------+----------+----------+------------------+
|       1| 1072.0|     P9935|Electronics|   Laptop|    3733.6| 3733.6| 14934.4|       4| North|     Low|Processing|2025-04-25| 4405.647999999999|
|       2|   NULL|     P4257|Electronics|   Laptop|    559.11| 559.11| 2236.44|       4|  West|    High|Processing|2025-09-16| 659.7497999999999|
|       4| 1310.0|     P1711|Electronics|   Laptop|   1961.64|1961.64|11769.84|       6|  West|     Low| Completed|2025-06-26|         2314.7352|
|      11| 1082.0|     P7916|Electronics|   Laptop|    4319.4| 4319.4| 12958.2|       3| North|  Medium|Processing|2025-11-1

## Step 9: Filtering with an AND condition
This time I wanted only Completed orders where the amount is more than 1000, so both conditions need to be true together.


In [0]:
completed_high_value = df.filter((col("status") == "Completed") & (col("amount") > 1000))
completed_high_value.show(5)
print("Completed & amount>1000:", completed_high_value.count())


+--------+-------+----------+-----------+----------+----------+-------+--------+--------+------+--------+---------+----------+------------------+
|order_id|user_id|product_id|   category| item_name|base_price|  price|  amount|quantity|region|priority|   status|order_date|       final_price|
+--------+-------+----------+-----------+----------+----------+-------+--------+--------+------+--------+---------+----------+------------------+
|       4| 1310.0|     P1711|Electronics|    Laptop|   1961.64|1961.64|11769.84|       6|  West|     Low|Completed|2025-06-26|         2314.7352|
|       8| 1118.0|     P6168|   Clothing|    Hoodie|   1176.11|1176.11| 7056.66|       6|  West|  Medium|Completed|2025-01-29|1387.8097999999998|
|      13| 1489.0|     P5315|Electronics|    Tablet|   2230.35|2230.35| 17842.8|       8| South|     Low|Completed|2025-01-02|2631.8129999999996|
|      18| 1302.0|     P4608|Electronics|Smartphone|   1201.02|1201.02| 7206.12|       6| North|    High|Completed|2025-01-1

## Step 10: Filtering with an OR condition
Here either condition can be true - region is North, or priority is High.


In [0]:
north_or_high = df.filter((col("region") == "North") | (col("priority") == "High"))
north_or_high.show(5)
print("North or High priority orders:", north_or_high.count())


+--------+-------+----------+-----------+-------------+----------+-------+-------+--------+------+--------+----------+----------+------------------+
|order_id|user_id|product_id|   category|    item_name|base_price|  price| amount|quantity|region|priority|    status|order_date|       final_price|
+--------+-------+----------+-----------+-------------+----------+-------+-------+--------+------+--------+----------+----------+------------------+
|       1| 1072.0|     P9935|Electronics|       Laptop|    3733.6| 3733.6|14934.4|       4| North|     Low|Processing|2025-04-25| 4405.647999999999|
|       2|   NULL|     P4257|Electronics|       Laptop|    559.11| 559.11|2236.44|       4|  West|    High|Processing|2025-09-16| 659.7497999999999|
|       3| 1358.0|     P5552|     Sports|Tennis Racket|   4066.21|4066.21|4066.21|       1| South|    High| Cancelled|2025-03-23|         4798.1278|
|       5| 1296.0|     P2139|  Groceries|         Rice|    2805.0| 2805.0|28050.0|      10| North|     Low

## Step 11: Handling Null values
Since I intentionally left some nulls in user_id (and a few other columns), let's check and handle them.


In [0]:
# how many nulls in user_id
print("Nulls in user_id:", df.filter(col("user_id").isNull()).count())

# option 1 - drop rows where user_id is null
df_dropped_nulls = df.na.drop(subset=["user_id"])

# option 2 - fill nulls with a default value instead of dropping
df_filled_nulls = df.na.fill({"user_id": 0})

print("Rows before:", df.count())
print("Rows after dropping null user_id:", df_dropped_nulls.count())


Nulls in user_id: 52
Rows before: 2020
Rows after dropping null user_id: 1968


## Step 12: Removing duplicate rows
I added a few duplicate rows on purpose in the dataset, so let's check and remove them.


In [0]:
print("Rows before removing duplicates:", df.count())
df_no_dupes = df.dropDuplicates()
print("Rows after removing duplicates:", df_no_dupes.count())


Rows before removing duplicates: 2020
Rows after removing duplicates: 2000


### Transformations vs Actions
select(), filter(), withColumn() and groupBy() are all transformations - Spark just builds up a plan, nothing actually runs. It's only when I call an action like show() or count() that Spark goes ahead and executes everything. In the cell below, groupBy and orderBy are transformations, and show() is the action that finally triggers it.


In [0]:
category_summary = df.groupBy("category").count().orderBy("count", ascending=False)
category_summary.show()


+-----------+-----+
|   category|count|
+-----------+-----+
|Electronics|  417|
|     Sports|  406|
|  Furniture|  403|
|   Clothing|  397|
|  Groceries|  382|
|       NULL|   15|
+-----------+-----+



## Step 14: Saving as CSV and Parquet
Saving the same data in both formats so I can compare them.


In [0]:
# Save as CSV
df.write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/datasets/orders_csv")

# Save as Parquet
df.write.mode("overwrite").parquet("/Volumes/workspace/default/datasets/orders_parquet")

### CSV vs Parquet
When I checked the folder sizes using `%fs ls`, the Parquet folder was noticeably smaller than the CSV folder even though it holds the exact same data. That's because CSV stores everything row by row as plain text, while Parquet stores it column by column and compresses each column. Parquet also carries its schema with it, CSV does not.


### Predicate Pushdown
When I filter on a Parquet file below, Spark doesn't load the whole file and then filter afterwards - it pushes the filter condition down to the storage layer, so it only reads the row groups that could actually match. This is one of the reasons filtering on Parquet is faster than on CSV, especially on bigger files.


In [0]:
electronics_parquet = spark.read.parquet("/Volumes/workspace/default/datasets/orders_parquet").filter(col("category") == "Electronics")
electronics_parquet.show(5)




+--------+-------+----------+-----------+---------+----------+-------+--------+--------+------+--------+----------+----------+------------------+
|order_id|user_id|product_id|   category|item_name|base_price|  price|  amount|quantity|region|priority|    status|order_date|       final_price|
+--------+-------+----------+-----------+---------+----------+-------+--------+--------+------+--------+----------+----------+------------------+
|       1| 1072.0|     P9935|Electronics|   Laptop|    3733.6| 3733.6| 14934.4|       4| North|     Low|Processing|2025-04-25| 4405.647999999999|
|       2|   NULL|     P4257|Electronics|   Laptop|    559.11| 559.11| 2236.44|       4|  West|    High|Processing|2025-09-16| 659.7497999999999|
|       4| 1310.0|     P1711|Electronics|   Laptop|   1961.64|1961.64|11769.84|       6|  West|     Low| Completed|2025-06-26|         2314.7352|
|      11| 1082.0|     P7916|Electronics|   Laptop|    4319.4| 4319.4| 12958.2|       3| North|  Medium|Processing|2025-11-1

## Step 16: Reading Parquet, filtering, and writing the final CSV
Final step of the pipeline - read the parquet file back, drop rows with null user_id, and write the result out as CSV.


In [0]:
final_df = spark.read.parquet("/Volumes/workspace/default/datasets/orders_parquet")

final_df = final_df.filter(col("user_id").isNotNull())

final_df.write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/datasets/final_orders_csv")

print("Final row count after removing null user_id:", final_df.count())

final_df.show(5)

Final row count after removing null user_id: 1968
+--------+-------+----------+-----------+-------------+----------+-------+--------+--------+------+--------+----------+----------+------------------+
|order_id|user_id|product_id|   category|    item_name|base_price|  price|  amount|quantity|region|priority|    status|order_date|       final_price|
+--------+-------+----------+-----------+-------------+----------+-------+--------+--------+------+--------+----------+----------+------------------+
|       1| 1072.0|     P9935|Electronics|       Laptop|    3733.6| 3733.6| 14934.4|       4| North|     Low|Processing|2025-04-25| 4405.647999999999|
|       3| 1358.0|     P5552|     Sports|Tennis Racket|   4066.21|4066.21| 4066.21|       1| South|    High| Cancelled|2025-03-23|         4798.1278|
|       4| 1310.0|     P1711|Electronics|       Laptop|   1961.64|1961.64|11769.84|       6|  West|     Low| Completed|2025-06-26|         2314.7352|
|       5| 1296.0|     P2139|  Groceries|         

### Best Practices - avoiding collect() on big data
collect() pulls all the data back to the driver, which can crash it once the dataset gets large. show(), limit() and take() are safer when I just want to peek at a few rows instead of bringing everything back.


In [0]:
# good practice - just look at a few rows
df.show(5)
df.limit(5).show()

# avoid this on a real big dataset, it pulls everything to the driver
# df.collect()


+--------+-------+----------+-----------+-------------+----------+-------+--------+--------+------+--------+----------+----------+------------------+
|order_id|user_id|product_id|   category|    item_name|base_price|  price|  amount|quantity|region|priority|    status|order_date|       final_price|
+--------+-------+----------+-----------+-------------+----------+-------+--------+--------+------+--------+----------+----------+------------------+
|       1| 1072.0|     P9935|Electronics|       Laptop|    3733.6| 3733.6| 14934.4|       4| North|     Low|Processing|2025-04-25| 4405.647999999999|
|       2|   NULL|     P4257|Electronics|       Laptop|    559.11| 559.11| 2236.44|       4|  West|    High|Processing|2025-09-16| 659.7497999999999|
|       3| 1358.0|     P5552|     Sports|Tennis Racket|   4066.21|4066.21| 4066.21|       1| South|    High| Cancelled|2025-03-23|         4798.1278|
|       4| 1310.0|     P1711|Electronics|       Laptop|   1961.64|1961.64|11769.84|       6|  West| 

## Conclusion
In this assignment I created my own orders dataset with nulls, duplicates and mixed datatypes, then used PySpark on Databricks to read it (both with inferSchema and a custom schema), explore it, rename and cast columns, add a derived column, filter data using AND/OR conditions, handle nulls and duplicates, compare transformations vs actions, and finally compare CSV vs Parquet storage along with predicate pushdown. Along the way I also went through the Spark architecture side - Driver, Executors, Cluster Manager, Client vs Cluster mode, Lazy Evaluation, Fault Tolerance and Shuffle - which helped connect the theory to what was actually happening when the code ran.
